In [ ]:
document = """
Refunds are processed within 5 to 7 business days.
International orders may take up to 14 days for refunds.
To request a refund, visit our returns portal.
Our customer support is available 24/7.
You can reset your password from the login page.
For technical issues, contact our helpdesk.
"""

# Split the document into chunks (by sentence)
chunks = [sentence.strip() for sentence in document.strip().split("\n") if sentence.strip()]

# Let's see what we got
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert each chunk into a vector
embeddings = model.encode(chunks)

# Let's see what we got
for i, embedding in enumerate(embeddings):
    print(f"Chunk {i+1}: {embedding[:5]}...")  # showing only first 5 numbers
    print(f"Vector size: {len(embedding)}")
    print()

In [ ]:
# Our question
question = "How long does a refund take?"

# Convert question to a vector
question_embedding = model.encode([question])[0]

print(f"Question vector (first 5): {question_embedding[:5]}")
print(f"Vector size: {len(question_embedding)}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compare question vector with all chunk vectors
similarities = cosine_similarity([question_embedding], embeddings)[0]

# Print similarity score for each chunk
for i, score in enumerate(similarities):
    print(f"Chunk {i+1}: {score:.4f} → {chunks[i]}")

# Find the most similar chunk
best_match = np.argmax(similarities)
print(f"\n✅ Best match: Chunk {best_match+1}")
print(f"→ {chunks[best_match]}")

In [ ]:
import chromadb

# Create a ChromaDB client (runs locally on your laptop)
client = chromadb.Client()

# Create a collection (like a table in regular database)
collection = client.create_collection(name="my_docs")

# Store all chunks with their embeddings
collection.add(
    documents=chunks,                          # original text
    embeddings=embeddings.tolist(),            # vectors
    ids=[f"chunk{i}" for i in range(len(chunks))]  # unique id for each
)

print("✅ All chunks stored in ChromaDB!")
print(f"Total chunks stored: {collection.count()}")

In [ ]:
# Our question
question = "How long does a refund take?"

# Convert question to vector
question_embedding = model.encode([question])[0]

# Search ChromaDB
results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=2  # return top 2 most relevant chunks
)

# Show results
print("🔍 Question:", question)
print()
print("✅ Most relevant chunks found:")
for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i+1}: {doc}")

In [10]:
import requests

# Step 1: Build the prompt
context = "\n".join(results["documents"][0])

prompt = f"""You are a helpful assistant.
Use the following context to answer the question.

Context:
{context}

Question: {question}

Answer:"""

# Step 2: Send to Ollama
response = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "llama3.2", "prompt": prompt, "stream": False},
)

# Step 3: Print the answer
answer = response.json()["response"]
print("Prompt: ", prompt)
print("🤖 Answer:", answer)

Prompt:  You are a helpful assistant.
Use the following context to answer the question.

Context:
Refunds are processed within 5 to 7 business days.
International orders may take up to 14 days for refunds.

Question: How long does a refund take?

Answer:
🤖 Answer: The processing time for refunds can vary depending on whether it's an international order or not. 

If the order is domestic, refunds are typically processed within 5 to 7 business days. 

However, if the order is international, refunds may take up to 14 business days to process.
